# 3. Dimensionality reduction, and how much it distorted

*Part of the [broadlisten](https://github.com/InteractionDataLab/broad-listening/tree/main/code)
stage-by-stage examples. Every number and figure below was produced by running
this notebook on the synthetic corpus in `broadlisten.sample`, which is
generated from templates and contains no participant data.*

Clustering high-dimensional vectors directly is hard, so they are projected
first. That projection is not free: it moves points, and clusters found in the
projection can be structure the projection invented.

So this stage returns a number for how much it distorted, rather than leaving
you to hope it was small.

In [1]:
import numpy as np
from broadlisten.sample import sample_corpus, sample_embeddings

corpus = sample_corpus(seed=0)
vectors = sample_embeddings(corpus.planted_theme.tolist(), seed=0)
print(f"{vectors.shape[0]} responses in {vectors.shape[1]} dimensions")

600 responses in 64 dimensions


In [2]:
from broadlisten.reduce import umap_project, faithfulness

reduced = umap_project(vectors, n_components=5, n_neighbors=25,
                       min_dist=0.0, seed=42)
print(f"reduced to {reduced.shape[1]} dimensions")
print(faithfulness(vectors, reduced, k=15))

...\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


reduced to 5 dimensions
Faithfulness(trustworthiness=0.942, continuity=0.942, mean=0.942)


**Trustworthiness** asks whether points that are close in the
projection were close before. **Continuity** asks the reverse: whether points
that were close stayed close. They fail differently, so both are reported.

## Determinism is not automatic

UMAP is reproducible only with a seed *and* `n_jobs=1`. `umap_project` pins both
and does not offer to unpin them, because a figure nobody can reproduce is not
evidence.

In [3]:
again = umap_project(vectors, n_components=5, n_neighbors=25,
                    min_dist=0.0, seed=42)
print("identical to the first run:", np.array_equal(reduced, again))

other = umap_project(vectors, n_components=5, n_neighbors=25,
                     min_dist=0.0, seed=7)
print("identical under a different seed:", np.array_equal(reduced, other))

identical to the first run: True


identical under a different seed: False


## Choosing `n_neighbors` by evidence

Faithfulness rises with `n_neighbors` while the local detail that clustering
needs falls, so the choice is a trade-off you should see rather than a default
you inherit.

In [4]:
from broadlisten.reduce import sweep_parameters

sweep = sweep_parameters(vectors, neighbors=(10, 25, 50),
                         min_dists=(0.0, 0.1), n_components=5, seed=42)
sweep

    n_neighbors=10   min_dist=0.0   trust=0.935 cont=0.935 mean=0.935


    n_neighbors=10   min_dist=0.1   trust=0.938 cont=0.939 mean=0.938


    n_neighbors=25   min_dist=0.0   trust=0.942 cont=0.942 mean=0.942


    n_neighbors=25   min_dist=0.1   trust=0.941 cont=0.944 mean=0.942


    n_neighbors=50   min_dist=0.0   trust=0.943 cont=0.941 mean=0.942


    n_neighbors=50   min_dist=0.1   trust=0.944 cont=0.943 mean=0.943


,n_neighbors,min_dist,n_components,trustworthiness,continuity,mean
0,50,0.1,5,0.943639,0.942650,0.943144
1,25,0.1,5,0.941142,0.943619,0.942381
2,25,0.0,5,0.942108,0.942301,0.942204
3,50,0.0,5,0.942623,0.940527,0.941575
4,10,0.1,5,0.937539,0.938654,0.938097
5,10,0.0,5,0.934940,0.934977,0.934958


**Next:** [4. Clustering](04_cluster.ipynb), where two methods disagree
on purpose.